# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) accessible from:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

This notebook explores its structure (record sets, fields, columns) and demonstrates how to load and analyze it programmatically.

In [ ]:
# Ensure mlcroissant is installed in the environment
!pip install mlcroissant

## 1. Data Loading

Let's load the Croissant schema and inspect the dataset metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Let's enumerate the available record sets, fields, and their associated `@id`s (identifiers) as described in the Croissant metadata.

This will help us understand the dataset's tables and schema.

In [ ]:
# List available record sets, their fields and columns with their @id
print("Available Record Sets (by @id):\n")
record_sets = dataset.record_sets

if not record_sets:
    # If the main .record_set list is empty, infer from 'distribution' (common in FAIR v1 croissant)
    print("record_sets is empty, inferring from distributions...")
    if hasattr(metadata, 'distribution'):
        for i, dist in enumerate(metadata.distribution):
            rid = dist['@id'] if '@id' in dist else f'distribution_{i}'
            print(f" - Distribution [{i}]: @id = {rid}")
        print('\nTo enumerate fields and columns, see the Croissant JSON or further dataset introspection below.')
    else:
        print("No distributions found.")
else:
    for record_set in record_sets:
        print(f"- RecordSet @id: {record_set['@id']}")
        print(f"  Name: {record_set.get('name', '<no name>')}")
        # List fields
        if 'field' in record_set:
            for field in record_set['field']:
                print(f"    Field @id: {field['@id']}  Name: {field.get('name', '<no name>')}  Type: {field.get('dataType', '<unknown>')}")
        print()

print('\nA more detailed view can be achieved by programmatically loading one record set (see next section).')

## 3. Data Extraction

Let's load tabular data from the first record set (or distribution file), as discovered above, into a Pandas DataFrame for further analysis.
All entities (record sets, fields, columns) must be referenced by their `@id`.

In [ ]:
# Since dataset.record_sets is empty in this schema, use the first distribution's @id as the record set
if hasattr(metadata, 'distribution') and len(metadata.distribution) > 0:
    # Take first distribution
    main_table_id = metadata.distribution[0]['@id']
    print(f"Main data record set (distribution) detected: {main_table_id}")
else:
    raise ValueError("No record sets or distributions found in dataset metadata.")

# Extract the records
records = list(dataset.records(record_set=main_table_id))
df = pd.DataFrame(records)

print(f"Data columns (fields):\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field for basic analysis, referencing it by its Croissant `@id`/column name as loaded above.

- We'll filter for records where a numeric field (e.g., 'MW [kDa]') is above a threshold.
- We'll normalize this field and group data by a categorical attribute if relevant.

In [ ]:
# Example field IDs (from prior df.columns, which are field @id or field name)
all_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not all_numeric_fields:
    # Heuristic: try MW, Abundance, or 'Count' fields by substring
    candidates = [col for col in df.columns if any(token in col.lower() for token in ['mw', 'abundance', 'count', 'peptide', 'area'])]
    print(f"No auto-detected numeric fields, but possible numeric columns: {candidates}")
    # Try to coerce one column as float for demonstration
    numeric_field = candidates[0] if candidates else df.columns[0]
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
else:
    numeric_field = all_numeric_fields[0]
print(f"Using field for numeric EDA: {numeric_field}")

# Remove non-numeric rows
filtered_df = df[df[numeric_field] > 10] if df[numeric_field].dtype in ['float64', 'int64'] else df

print(f"Filtered records with '{numeric_field}' > 10:")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by closest categorical field
group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and col != numeric_field]
group_field = group_fields[0] if group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean '{numeric_field}' by '{group_field}':")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field and group-wise aggregates.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution histogram
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field is set, boxplot
if group_field and group_field != numeric_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion

- We've loaded the dataset metadata and tabular data using `mlcroissant`, referencing all entities by their Croissant `@id`.
- Explored dataset fields and extracted records for basic analysis.
- Performed simple EDA: filtering, normalization, grouping, and visualization.

Explore further by referencing the [Croissant documentation](https://mlcommons.org/croissant/) and inspecting the dataset structure for additional fields, distributions, or record sets.

**Citation:**

Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.